In [1]:
import asyncio
import time

async def read_label() -> str:
    await asyncio.sleep(3)
    return "document-ready"

async def main() -> None:
    pendding = read_label()
    label = await pendding
    assert label == "document-ready"

if __name__ == "__main__":
    await main()

In [16]:
import asyncio
import time

async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what, time.strftime("%X"))

async def main():
    print(f"started at {time.strftime("%X")}")

    await say_after(1, "hello")
    await say_after(2, "world")

    print(f"finished at {time.strftime("%X")}")

if __name__ == "__main__":
    await main()

started at 11:23:50
hello 11:23:51
world 11:23:53
finished at 11:23:53


In [19]:
import asyncio
import time

async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what, time.strftime("%X"))

async def main():
    print(f"started at {time.strftime("%X")}")

    task1 = asyncio.create_task(
        say_after(2, "hello")
    )
    task2 = asyncio.create_task(
        say_after(1, "world")
    )

    await task1
    await task2
    
    print(f"finished at {time.strftime("%X")}")

if __name__ == "__main__":
    await main()

started at 11:26:32
world 11:26:33
hello 11:26:34
finished at 11:26:34


In [20]:
import asyncio
import time

async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what, time.strftime("%X"))

async def main():
    print(f"started at {time.strftime("%X")}")

    async with asyncio.TaskGroup() as tg:
        task1 = asyncio.create_task(
            say_after(2, "hello")
        )
        task2 = asyncio.create_task(
            say_after(1, "world")
        )
        
        print(f"started 2 at {time.strftime("%X")}")

    print(f"finished at {time.strftime("%X")}")

if __name__ == "__main__":
    await main()

started at 11:42:28
started 2 at 11:42:28
finished at 11:42:28
world 11:42:29
hello 11:42:30


In [22]:
import asyncio

async def nested():
    return 42

async def main():
    # nested()

    print(await nested())

if __name__ == "__main__":
    await main()

42


In [33]:
import asyncio, time

async def nested():
    print("nested start", time.time())
    return 42

async def main():
    print("main start", time.time())
    task = asyncio.create_task(nested())
    print("main start end", time.time())
    print(task, time.time())
    r = await task
    print(r)

if __name__ == "__main__":
    await main()

main start 1791691007.358572
main start end 1791691007.358631
<Task pending name='Task-240' coro=<nested() running at /var/folders/22/6rtxv1y92dvfc1_czm4pw77m0000gn/T/ipykernel_55108/3182081970.py:3>> 1791691007.358637
nested start 1791691007.3588
42


In [34]:
import asyncio
import datetime

async def display_date():
    loop = asyncio.get_running_loop()
    end_time = loop.time() + 5.0
    while True:
        print(datetime.datetime.now())
        if (loop.time() + 1.0) >= end_time:
            break
        await asyncio.sleep(1)

await display_date()

2026-10-11 12:20:04.461392
2026-10-11 12:20:05.462969
2026-10-11 12:20:06.464176
2026-10-11 12:20:07.465415
2026-10-11 12:20:08.466668


In [36]:
import asyncio

async def eternity():
    await asyncio.sleep(3600)
    print("yay!")

async def main():
    try:
        await asyncio.wait_for(eternity(), timeout=1.0)
    except TimeoutError:
        print("timeout!")

await main()

yay!


In [44]:
import asyncio, time

async def delayed_value(value: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return value

async def compare() -> None:
    print("start", time.time())
    first = await delayed_value("A", 1)
    second = await delayed_value("B", 2)
    print("end", time.time())
    print(first, second)
    assert [first, second] == ["A", "B"]

    print("start2", time.time())
    combined = await asyncio.gather(
        delayed_value("A", 1),
        delayed_value("B", 2)
    )
    print("end2", time.time())
    print(combined)
    assert combined == ["A", "B"]

await compare()

start 1791726555.681236
end 1791726558.6841621
A B
start2 1791726558.684337
end2 1791726560.685139
['A', 'B']


In [47]:
import httpx

def handler(request: httpx.Request) -> httpx.Response:
    assert request.url.path == "/status"
    return httpx.Response(200, json={"ready": True})

async def read_status() -> bool:
    transport = httpx.MockTransport(handler)
    async with httpx.AsyncClient(transport=transport) as client:
        response = await client.get(
            "https://training.invalid/status",
            timeout=5.0,
        )
        response.raise_for_status()
        payload = response.json()
        return payload["ready"]

r = await read_status()
print(r)

True


In [1]:
from typing import Protocol

class JsonTransport(Protocol):
    async def get_json(self, url: str, *, timeout: float) -> object:
        ...

class FakeTransport:
    def __init__(self, payload: object) -> None:
        self.payload = payload
        self.calls: list[tuple[str, float]] = []

    async def get_json(self, url: str, *, timeout: float) -> object:
        self.calls.append((url, timeout))
        return self.payload

async def use_transport(transport: JsonTransport) -> object:
    return await transport.get_json("https://training.invalid/status", timeout=5.0)